# Banking Fraud Intelligence
## Transaction history and chronological splits

**P2 · Feature engineering · Simulated card transactions**

**Question.** What customer and terminal activity would we know when a new transaction arrives?

**Approach.** Use the same validated CSV from [the exploratory notebook](01_fraud_investigation_eda.ipynb). Build six history features, check them against a worked example, and verify the chronological partitions. The notebook uses the same [Python functions](../src/transaction_features.py) that the modelling step will call.

Open this notebook from the repo in VS Code, select the Python 3.12 kernel that ran the first notebook, then choose **Run All**. It uses `data/processed/transactions.csv`. [Local setup](../data/README.md#run-the-feature-engineering-notebook)

## 1 · Define the scoring inputs

| Feature | Meaning |
|---|---|
| `TX_AMOUNT` | Amount of the current transaction |
| `customer_tx_count_24h`, `customer_tx_count_7d` | Previous transactions for this customer in each window |
| `terminal_tx_count_24h`, `terminal_tx_count_7d` | Previous transactions for this terminal in each window |
| `customer_mean_amount_7d` | Mean amount of the customer's previous transactions in the seven-day window |
| `customer_seconds_since_previous` | Elapsed seconds since the customer's latest strictly earlier transaction |

Each window includes its lower boundary and excludes the current timestamp. Counts start at zero. A missing prior mean or elapsed time stays missing when the required history is unavailable. Any imputation belongs to the training pipeline. Outcome and scenario fields stay available for evaluation, outside the predictor list.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

repo = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "transaction_features.py").is_file()),
    None,
)
if repo is None:
    raise FileNotFoundError("Open this notebook from the Banking Fraud Intelligence checkout.")

if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from src.transaction_features import MODEL_FEATURES, SPLIT_PERIODS, build_features, split_features
from src.validate_transactions import load_transactions

## 2 · See how equal timestamps work

These four invented transactions share a customer and terminal. Both 10:00 transactions see only the 09:00 transaction. The 10:01 transaction can see all three earlier events.

In [ ]:
example = pd.DataFrame({
    "TRANSACTION_ID": [1, 2, 3, 4],
    "TX_DATETIME": pd.to_datetime([
        "2018-04-01 09:00:00", "2018-04-01 10:00:00",
        "2018-04-01 10:00:00", "2018-04-01 10:01:00",
    ]),
    "CUSTOMER_ID": [1, 1, 1, 1],
    "TERMINAL_ID": [10, 10, 10, 10],
    "TX_AMOUNT": [40.0, 80.0, 120.0, 200.0],
    "TX_FRAUD": [0, 0, 0, 0],
    "TX_FRAUD_SCENARIO": [0, 0, 0, 0],
})
example_features = build_features(example)
assert example_features["customer_tx_count_24h"].tolist() == [0, 1, 1, 3]
assert example_features["terminal_tx_count_24h"].tolist() == [0, 1, 1, 3]
assert example_features["customer_mean_amount_7d"].iloc[1:].tolist() == [40.0, 40.0, 80.0]
pd.testing.assert_frame_equal(
    build_features(example.iloc[:2]),
    example_features.iloc[:2],
)
display(example_features[[
    "TX_DATETIME", "TX_AMOUNT", "customer_tx_count_24h",
    "terminal_tx_count_24h", "customer_mean_amount_7d",
    "customer_seconds_since_previous",
]])
print("Equal-time and future-event checks passed.")

## 3 · Build features on the full CSV

Validate the source, then calculate features in event order. The checks reconcile transaction IDs and confirm the expected relationships between the two window lengths. Missing history is expected, especially at the start of the dataset.

In [ ]:
transactions = load_transactions()
features = build_features(transactions)
pd.testing.assert_series_equal(features["TRANSACTION_ID"], transactions["TRANSACTION_ID"])
assert features["TRANSACTION_ID"].is_unique
for entity in ("customer", "terminal"):
    assert (features[f"{entity}_tx_count_24h"] <= features[f"{entity}_tx_count_7d"]).all()
assert (features["customer_seconds_since_previous"].dropna() > 0).all()
assert (
    features["customer_mean_amount_7d"].isna() == features["customer_tx_count_7d"].eq(0)
).all()

print(f"{len(features):,} feature rows and {len(MODEL_FEATURES)} model inputs, including the current amount.")
display(features[list(MODEL_FEATURES)].isna().sum().rename("Missing rows").to_frame())

## 4 · Verify the date boundaries

Train on April through July, choose the model on August, and reserve September for final evaluation. Observed transactions before a cutoff still supply history for transactions after it. Recompute the training features using only training-period events and verify that the values exactly match those from the full build.

In [ ]:
partitions = split_features(features)
summary = pd.DataFrame.from_records([
    {
        "Partition": name.capitalize(),
        "Start (inclusive)": SPLIT_PERIODS[name][0].date(),
        "End (exclusive)": SPLIT_PERIODS[name][1].date(),
        "Transactions": f"{len(partition):,}",
    }
    for name, partition in partitions.items()
])
pd.testing.assert_series_equal(
    pd.concat([partition["TRANSACTION_ID"] for partition in partitions.values()]),
    features["TRANSACTION_ID"],
)
training_source = transactions.loc[partitions["train"].index]
pd.testing.assert_frame_equal(build_features(training_source), partitions["train"])
display(summary.set_index("Partition"))
print("Complete partition coverage and exact training-feature reproducibility verified.")

## Next checkpoint

Use the seven model inputs and the training partition to fit the first model. Compare its August rankings with the amount-based baseline at the same daily review limits. The final September evaluation follows model selection. This notebook builds the feature table in memory and shows synthetic examples and aggregate diagnostics. [Project handoff](../docs/SESSION_HANDOFF_CURRENT.md)